In [9]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from xgboost import XGBClassifier

In [10]:
df = pd.read_csv('../data/student-mat.csv', sep=';')

df['at_risk'] = (df['G3'] < 10).astype(int)

df.head()

,school,sex,age,address,famsize,Pstatus,Medu,Fedu,Mjob,Fjob,...,freetime,goout,Dalc,Walc,health,absences,G1,G2,G3,at_risk
0,GP,F,18,U,GT3,A,4,4,at_home,teacher,...,3,4,1,1,3,6,5,6,6,1
1,GP,F,17,U,GT3,T,1,1,at_home,other,...,3,3,1,1,3,4,5,5,6,1
2,GP,F,15,U,LE3,T,1,1,at_home,other,...,3,2,2,3,3,10,7,8,10,0
3,GP,F,15,U,GT3,T,4,2,health,services,...,2,2,1,1,5,2,15,14,15,0
4,GP,F,16,U,GT3,T,3,3,other,other,...,3,2,1,2,5,4,6,10,10,0


In [11]:
FEATURES = [

    'school',
    'sex',
    'age',
    'address',
    'famsize',
    'Pstatus',

    'Medu',
    'Fedu',

    'Mjob',
    'Fjob',

    'reason',
    'guardian',

    'traveltime',
    'studytime',
    'failures',

    'schoolsup',
    'famsup',
    'paid',
    'activities',

    'higher',
    'internet',
    'romantic',

    'famrel',
    'freetime',
    'goout',

    'Dalc',
    'Walc',

    'health',
    'absences'
]

X = df[FEATURES]
y = df['at_risk']

In [12]:
categorical_features = X.select_dtypes(
    include=['object']
).columns

numeric_features = X.select_dtypes(
    exclude=['object']
).columns

preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(handle_unknown='ignore'),
            categorical_features
        ),
        (
            'num',
            'passthrough',
            numeric_features
        )
    ]
)

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [14]:
model = XGBClassifier(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric='logloss',
    random_state=42
)

In [15]:
pipeline = Pipeline([
    ('prep', preprocessor),
    ('model', model)
])

pipeline.fit(X_train, y_train)

,steps,"[('prep', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('cat', ...), ('num', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


In [16]:
y_pred = pipeline.predict(X_test)

y_prob = pipeline.predict_proba(X_test)[:,1]

print("Accuracy :", accuracy_score(y_test,y_pred))
print("Precision:", precision_score(y_test,y_pred))
print("Recall   :", recall_score(y_test,y_pred))
print("F1       :", f1_score(y_test,y_pred))
print("ROC AUC  :", roc_auc_score(y_test,y_prob))

Accuracy : 0.7088607594936709
Precision: 0.5882352941176471
Recall   : 0.38461538461538464
F1       : 0.46511627906976744
ROC AUC  : 0.7213352685050798


In [17]:
print(df["at_risk"].value_counts())
print(df["at_risk"].value_counts(normalize=True))

at_risk
0    265
1    130
Name: count, dtype: int64
at_risk
0    0.670886
1    0.329114
Name: proportion, dtype: float64


In [19]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

categorical = X.select_dtypes(include='object').columns
numeric = X.select_dtypes(exclude='object').columns

preprocessor = ColumnTransformer(
    [
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical
        ),
        (
            "num",
            "passthrough",
            numeric
        )
    ]
)

rf = RandomForestClassifier(
    n_estimators=500,
    random_state=42
)

pipe = Pipeline([
    ("prep", preprocessor),
    ("rf", rf)
])

scores = cross_val_score(
    pipe,
    X,
    y,
    cv=5,
    scoring="roc_auc"
)

print("ROC-AUC scores:", scores)
print("Mean ROC-AUC:", scores.mean())

ROC-AUC scores: [0.67416546 0.78918723 0.57184325 0.66400581 0.62953556]
Mean ROC-AUC: 0.6657474600870826
